In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



In [3]:
df = pd.read_csv("flights_sample_3m.csv")
print(df.shape)
print(df.isnull().sum())
print(df.columns.to_list())

(3000000, 32)
FL_DATE                          0
AIRLINE                          0
AIRLINE_DOT                      0
AIRLINE_CODE                     0
DOT_CODE                         0
FL_NUMBER                        0
ORIGIN                           0
ORIGIN_CITY                      0
DEST                             0
DEST_CITY                        0
CRS_DEP_TIME                     0
DEP_TIME                     77615
DEP_DELAY                    77644
TAXI_OUT                     78806
WHEELS_OFF                   78806
WHEELS_ON                    79944
TAXI_IN                      79944
CRS_ARR_TIME                     0
ARR_TIME                     79942
ARR_DELAY                    86198
CANCELLED                        0
CANCELLATION_CODE          2920860
DIVERTED                         0
CRS_ELAPSED_TIME                14
ELAPSED_TIME                 86198
AIR_TIME                     86198
DISTANCE                         0
DELAY_DUE_CARRIER          2466137
DELAY_

In [4]:
cols = ['FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'DEP_DELAY', 'CRS_ARR_TIME', 'DISTANCE', 'CRS_ELAPSED_TIME', 'CANCELLED']
df_train = df[cols].copy()


In [5]:
#Drop cancelled flights first and then the column
df_train = df_train[df_train['CANCELLED'] == 0]
df_train = df_train.drop('CANCELLED', axis=1)

In [6]:
#Drop Nan valued DEP_DELAY rows
df_train = df_train.dropna(subset=['DEP_DELAY'])
print(df_train.shape)

print(df_train.isnull().sum())

(2920860, 9)
FL_DATE             0
AIRLINE             0
ORIGIN              0
DEST                0
CRS_DEP_TIME        0
DEP_DELAY           0
CRS_ARR_TIME        0
DISTANCE            0
CRS_ELAPSED_TIME    0
dtype: int64


In [7]:
#Keeping the delays between 1 to 99 percentile range and ignoring the rest as outliers
upper = df_train['DEP_DELAY'].quantile(0.99)
lower = df_train['DEP_DELAY'].quantile(0.01)
df_train = df_train[df_train['DEP_DELAY'].between(lower, upper)]
print(df_train.shape)

(2865193, 9)


In [8]:
#1. Extract Data features
df_train['FL_DATE'] = pd.to_datetime(df_train['FL_DATE'])
df_train['day_of_week'] = df_train['FL_DATE'].dt.dayofweek
df_train['month'] = df_train['FL_DATE'].dt.month
df_train['dep_hour'] = df_train['CRS_DEP_TIME'] // 100
df_train['arr_hour'] = df_train['CRS_ARR_TIME'] // 100

print(df_train.shape)

(2865193, 13)


In [9]:
#count departures per hour at airport
dep_counts = df_train.groupby(['FL_DATE', 'ORIGIN', 'dep_hour']).size().reset_index(name='dep_count')
#count arrivals per hour at airport
arr_counts = df_train.groupby(['FL_DATE', 'DEST', 'arr_hour']).size().reset_index(name='arr_count')

In [10]:
df_train = df_train.merge(dep_counts, on=['FL_DATE', 'ORIGIN', 'dep_hour'], how='left')
df_train = df_train.merge(arr_counts, on=['FL_DATE', 'DEST', 'arr_hour'], how='left')


In [11]:
df_train = df_train.drop(['FL_DATE', 'CRS_DEP_TIME', 'CRS_ARR_TIME'], axis=1)
print(df_train.shape)
print(df_train.columns.to_list())

(2865193, 12)
['AIRLINE', 'ORIGIN', 'DEST', 'DEP_DELAY', 'DISTANCE', 'CRS_ELAPSED_TIME', 'day_of_week', 'month', 'dep_hour', 'arr_hour', 'dep_count', 'arr_count']


In [12]:
top_origins = df_train['ORIGIN'].value_counts().nlargest(50).index
top_destinations = df_train['DEST'].value_counts().nlargest(50).index
df_train['ORIGIN'] = df_train['ORIGIN'].where(df_train['ORIGIN'].isin(top_origins), 'OTHER')
df_train['DEST'] = df_train['DEST'].where(df_train['DEST'].isin(top_destinations), 'OTHER')


In [13]:
df_encoded = pd.get_dummies(df_train, ['AIRLINE', 'ORIGIN', 'DEST'], drop_first=True)
df_sample = df_encoded.sample(n = 300000, random_state=42)
X = df_sample.drop('DEP_DELAY',axis=1)
y = df_sample['DEP_DELAY']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)

(240000, 125)


In [14]:
lr_delay = LinearRegression()
lr_delay.fit(X_train, y_train)
y_pred_lr = lr_delay.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred_lr)
print(f"Linear Regression RMSE:, {mse_lr ** 0.5:.2f}")

Linear Regression RMSE:, 26.59


In [15]:
print(df_sample['dep_count'].describe())

count    300000.000000
mean          3.213540
std           2.506492
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          20.000000
Name: dep_count, dtype: float64


In [16]:
print(df_sample['arr_count'].describe())

count    300000.000000
mean          3.123293
std           2.521826
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          20.000000
Name: arr_count, dtype: float64


In [17]:
print(df_sample[['dep_count', 'arr_count', 'DEP_DELAY']].corr())

           dep_count  arr_count  DEP_DELAY
dep_count   1.000000  -0.217061   0.017036
arr_count  -0.217061   1.000000  -0.021779
DEP_DELAY   0.017036  -0.021779   1.000000


In [18]:
xgb_delay = xgb.XGBRegressor()
xgb_delay.fit(X_train, y_train)
y_pred_xgb = xgb_delay.predict(X_test)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
print(f"XGBoost RMSE:, {mse_xgb ** 0.5:.2f}")

XGBoost RMSE:, 26.44
